# UMUD Quick and Dirty

This notebooks shows
- how to find the resolution of all test images in pixels per cm,
- how to quickly locate the two aponeuroses in an ultrasound image (without machine learning), and
- how to determine the slope of the fascicles (again without machine learning).

Whatever other algorithm you're implementing, you'll need to know the pixels per cm of every image, and you can take these values from here.

There are 309 test images, and the notebook doesn't look at the training data.

References
- https://www.kaggle.com/competitions/umud-challenge-muscle-architecture-in-ultrasound-data


In [ ]:
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from glob import glob
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import math
from tqdm import tqdm
import warnings

competition_dir = '/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data'

# Collect the metadata

For all images, we determine the resolution in pixels per cm and the bounding box of the real image. The code distinguishes various classes of image (depending on the ultrasound equipment which was used to generate the image) and measures the distance between tick marks on the x or y axis.

In [ ]:
def collect_metadata(pattern):
    """Determine px_per_cm and the bounding box for every image whose path matches pattern."""
    old_a = None
    nt = []
    test_df = []
    prev_dist_list = []
    for path in sorted(glob(f'{competition_dir}/{pattern}')):
        px_per_cm = -1
        l, t, r, b = -1, -1, -1, -1
        id = int(path[-8:-4])
        filetype = path[-3:]
        with Image.open(path) as img:
            a = np.asarray(img)
        if len(a.shape) == 3:
            assert a.shape[2] == 3
        height, width = a.shape[:2]

        # Determine the scale (pixels per cm)
        if filetype == 'png':
            # only in test dataset
            # variable scale (3..6 cm high)
            # images are 2.85 cm wide and have square pixels
            first_tick = np.argmax(a[:, 6].mean(axis=-1) > 50) # 117
            second_minor_tick = 150 + np.argmax(a[150:, 6].mean(axis=-1) > 50)
            second_major_tick = 150 + np.argmax(a[150:, 9].mean(axis=-1) > 50)
            last_tick = len(a) - 1 - np.argmax(a[:, 6].mean(axis=-1)[::-1] > 50)
            if (second_major_tick - first_tick) < 3 * (second_minor_tick - first_tick):
                px_per_cm = second_major_tick - first_tick # major ticks are 10 mm, minor ticks are 5 mm
            else:
                px_per_cm = second_minor_tick - first_tick # major ticks are 50 mm, minor ticks are 10 mm
            hw = a.shape[1] // 2
            w2 = np.argmin(a[:, hw:].sum(axis=(0, 2)))
            l, t, r, b = hw - w2, first_tick, hw + w2, last_tick
            assert 2.8 < (r - l) / px_per_cm < 2.9 # image width in cm
        elif a.shape == (800, 1200, 3):
            # tick marks can be left, right, and at the bottom; train images can be hflipped; pixels can be non-square
            if (a[87, 1147:1157] == 175).all():
                # variable scale (3..7 cm high), tick marks at right border
                # images in test dataset are 5.75 cm wide and have square pixels
                assert (a[87, 1147:1157] == 175).all() # first tick
                first_tick = np.argmax(a[:, 1150].mean(axis=-1) > 50) # 87
                # assert first_tick == 87, first_tick
                second_major_tick = first_tick + 20 + np.argmax(a[first_tick + 20:, 1150].mean(axis=-1) > 50)
                last_tick = len(a) - 1 - np.argmax(a[:, 1150].mean(axis=-1)[::-1] > 50)
                n_ticks = (last_tick - first_tick) / (second_major_tick - first_tick)
                nt.append(n_ticks)
                n_ticks = round(n_ticks)
                if n_ticks <= 14:
                    px_per_cm = (last_tick - first_tick) / n_ticks * 2 # <= 14 ticks -> 2 ticks per cm
                    if n_ticks == 7:
                        l, t, r, b = 142, 91, 1058, last_tick # 3.5 cm
                    elif n_ticks == 8:
                        l, t, r, b = 163, 91, 1037, last_tick # 4 cm
                    elif n_ticks == 9:
                        l, t, r, b = 211, 91, 989, last_tick # 4.5 cm
                    elif n_ticks == 10:
                        l, t, r, b = 249, 91, 951, last_tick # 5 cm
                    elif n_ticks == 11:
                        l, t, r, b = 282, 91, 918, last_tick # 5.5 cm
                    elif n_ticks == 12:
                        l, t, r, b = 308, 91, 892, last_tick # 6 cm
                    elif n_ticks == 13:
                        l, t, r, b = 331, 91, 869, last_tick # 6.5 cm
                    elif n_ticks == 14:
                        l, t, r, b = 349, 91, 851, last_tick # 7 cm
                    else:
                        assert False
                    assert 5.7 < (r - l) / px_per_cm < 5.8 # image width in cm
                elif n_ticks == 15:
                    px_per_cm = (last_tick - first_tick) / 3 # 15 ticks = 3 cm -> 5 ticks per cm
                    l, t, r, b = 142, 91, 1058, last_tick # 3 cm
                    assert 5.7 < (r - l) / px_per_cm < 5.8 # image width in cm
                else:
                    assert False
            elif  (a[42, 67:74, 0] > 115).all():
                # 42px = 0 cm, 783px = 5 cm, tick marks at left border
                px_per_cm = (783 - 42) / 5
                l, t, r, b = 171, 42, 1029, 798 # ticks left 5 cm
            else:
                assert False
        elif a.shape == (644, 1088, 3):
            # 0px = 0cm, 630.5px = 5 cm
            px_per_cm = 630.5 / 5
            l, t, r, b = 140, 0, 947, 643 # ticks left 5 cm
        elif a.shape[0] in [512, 513]:
            # fixed scale
            if a[-5, 49, 0] == 168 and a[-5, 438, 0] == 168:
                px_per_cm = (442 - 53) / 5
            elif a[-5, 52, 0] == 168 and a[-5, 441, 0] == 168:
                px_per_cm = (442 - 53) / 5
            elif a[-5, 53, 0] == 168 and a[-5, 442, 0] == 168:
                px_per_cm = (442 - 53) / 5
            else:
                assert False
            l, t, r, b = 0, 0, width, height - 10
        elif a.shape[0] == 853:
            assert len(a.shape) == 2
            if a[-5, 100] == 170 and a[-5, 934] == 170:
                px_per_cm = (934 - 100) / 5
            elif a[-5, 44] == 170 and a[-5, 879] == 170:
                px_per_cm = (879 - 44) / 5
            else:
                assert False
            l, t, r, b = 0, 0, width, height
        else:
            assert False, "Unknown format"
            
        test_df.append((id, 
                        path.split('/')[-1],
                        filetype,
                        a.shape[0], a.shape[1],
                        len(a.shape),
                        px_per_cm,
                        l, t, r, b))
        # print(f"{id:4}.{filetype}: {a.shape[0]:5} {a.shape[1]:5} {len(a.shape)} {px_per_cm:7.2f}")
    
    test_df = pd.DataFrame(test_df, columns=['id', 'image_id', 'filetype', 'height', 'width', 'dim', 'px_per_cm', 'l', 't', 'r', 'b'])
    print(f'good: {((test_df.px_per_cm > 0) & (test_df.l >= 0)).mean():.0%}')
    return test_df


In [ ]:
test_df = collect_metadata('test_images_v2/test_set_v2/IMG_*.*') # 309 images
test_df

In [ ]:
# collect_metadata('fasc_imgs_v1/fasc_images_new_model_v1/image_*.*') # 2761 images, not implemented
# collect_metadata('apo_imgs_v1/apo_images_new_model_v1/image_*.*') # 1048 images, not implemented

# Find the two aponeuroses and the fascicle slope

We determine the two brightest horizontal lines and assume that they are the two aponeuroses. Then we take a patch of muscle tissue between the aponeurose `s_left` and correlate it with patches displaced by (dx, dy) `s_right`. The (dx, dy) which gives the highest correlation leads to the slope of the fascicles.

We predict:
- muscle thickness is the distance of the two aponeuroses,
- pennation angle is the slope of the fascicles,
- fascicle length is muscle thickness divided by the sine of the pennation angle.

We show four annotated images where the heuristics work as designed. 

Your task: Plot some more annotated images, see where the heuristics fail, and improve them!

In [ ]:
%%time
for idx in np.arange(0, len(test_df), 1):
    # Read the image
    image_id = test_df.image_id.iloc[idx]
    path = f"{competition_dir}/test_images_v2/test_set_v2/{image_id}"
    with Image.open(path) as img:
        a = np.asarray(img)

    # Extract the real image
    b = a[test_df.iloc[idx].t:test_df.iloc[idx].b, test_df.iloc[idx].l:test_df.iloc[idx].r]
    if len(b.shape) == 3:
        b = b.mean(axis=-1)
    px_per_cm = test_df.iloc[idx].px_per_cm
    mm_per_px = 10 / px_per_cm
    total_depth_cm = b.shape[0] / px_per_cm

    # Estimate the depth of the aponeuroses
    # Muscle thickness is between 10 and 50mm
    b0 = b.mean(axis=1)
    b1 = savgol_filter(b0, 31, 3)
    superficial_depth_px = np.argmax(b1[:min(int(2.0 * px_per_cm), len(b1) - int(px_per_cm))])
    min_deep_depth_px = superficial_depth_px + int(px_per_cm)
    max_deep_depth_px = superficial_depth_px + 5 * int(px_per_cm)
    deep_depth_px = min_deep_depth_px + np.argmax(b1[min_deep_depth_px:max_deep_depth_px])
    mt_px = deep_depth_px - superficial_depth_px

    # Estimate the prediction angle between 5 and 45 degrees
    middle_depth_px = (superficial_depth_px + deep_depth_px) // 2
    dx = 25
    ly = min(50, mt_px // 2 - dx)
    # if ly < 50: print('ly', superficial_depth_px, deep_depth_px, ly)
    best_dys = []
    corr_lines = []
    for x in range(0, b.shape[1] - dx, 25):
        s_left = b[middle_depth_px - ly : middle_depth_px + ly, x]
        assert len(s_left) >= 2
        if s_left.var() == 0:
            continue # A patch without variance cannot be correlated
        best_corr, best_dy = -np.inf, None
        dys, corrs = [], []
        for dy in range(-dx, dx):
            s_right = b[middle_depth_px - ly + dy: middle_depth_px + ly + dy, x + dx]
            with warnings.catch_warnings():
                # warnings.filterwarnings('ignore', category=RuntimeWarning)
                corr = np.corrcoef(s_left, s_right)[1, 0]
            if np.isfinite(corr): # correlation can be nan if image has black areas
                dys.append(dy)
                corrs.append(corr)
                if corr > best_corr:
                    best_corr, best_dy = corr, dy
        corr_lines.append((dys, corrs))
        if best_dy is not None: 
            best_dys.append(best_dy)
    assert len(best_dys) >= 5
    best_dy = np.median(best_dys)
    pa_rad = np.abs(np.atan(best_dy / dx))
    pa_deg = (pa_rad / math.pi * 180).clip(None, 45)
    if pa_deg >= 5:
        fl_px = mt_px / np.sin(pa_rad)
    else: # fascicle angle determination failed
        pa_deg = 15
        fl_px = mt_px / 0.34
    
    test_df.at[idx, 'superficial_depth_mm'] = superficial_depth_px * mm_per_px
    test_df.at[idx, 'deep_depth_mm'] = deep_depth_px * mm_per_px
    test_df.at[idx, 'mt_mm'] = mt_px * mm_per_px
    test_df.at[idx, 'fl_mm'] = np.clip(fl_px * mm_per_px, 30, 200)
    test_df.at[idx, 'pa_deg'] = pa_deg

    # Plot a few predictions
    if idx in [1, 7, 58, 300]:
        # Plot the vertical cut and aponeurosis depths
        plt.figure(figsize=(6, 2))
        # plt.plot(b0)
        plt.plot(b1, label='brightness')
        plt.scatter([superficial_depth_px, deep_depth_px], [100, 100], s=25, c='r', label='aponeurosis')
        plt.title(f"{image_id}: Vertical cross-section")
        plt.legend()
        plt.xlabel('y')
        plt.ylabel('brightness')
        plt.show()

        # Plot the correlations which lead to the pennation angle
        for dys, corrs in corr_lines:
            plt.plot(dys, corrs, color='gray')
        for bdy in best_dys:
            plt.axvline(bdy)
        plt.axvline(best_dy, color='r')
        plt.xlabel('dy')
        plt.ylabel('corr')
        plt.show()
        
        # Plot the annotated image
        plt.figure(figsize=(12, 8))
        plt.imshow(a) # image
        
        width = test_df.iloc[idx].r - test_df.iloc[idx].l
        height = test_df.iloc[idx].b - test_df.iloc[idx].t
        rect = plt.Rectangle((test_df.iloc[idx].l, test_df.iloc[idx].t), width, height,
                             edgecolor="y", linewidth=3, fill=False) # bounding box
        plt.gca().add_patch(rect) # bounding box
        
        plt.hlines(test_df.iloc[idx].t + np.array([superficial_depth_px, deep_depth_px]),
                   test_df.iloc[idx].l,
                   test_df.iloc[idx].r,
                   linestyles='dashed',
                   color='w', label='aponeurosis') # superficial and deep aponeurosis

        if best_dy != 0:
            cx = test_df.iloc[idx].l + width // 2
            plt.plot(cx + np.array([-1, 1]) * dx / best_dy * mt_px / 2,
                     test_df.iloc[idx].t + np.array([superficial_depth_px, deep_depth_px]),
                     color='lightgreen') # fascicle
        
        plt.title(f"{image_id}: mt={test_df.at[idx, 'mt_mm']:.1f} mm, pa={test_df.at[idx, 'pa_deg']:.1f}°, fl={test_df.at[idx, 'fl_mm']:.1f} mm")
        plt.show()


# Submission

In [ ]:
assert not test_df[['pa_deg', 'fl_mm', 'mt_mm']].isna().any().any()
submission = test_df[['image_id', 'pa_deg', 'fl_mm', 'mt_mm']]
submission.to_csv('submission.csv', index=False)
!head submission.csv

In [ ]:
plt.hist(test_df.superficial_depth_mm, bins=15)
plt.title('Depth of superficial aponeurosis histogram')
plt.xlabel('depth in mm')
plt.ylabel('count')
plt.show()

In [ ]:
plt.hist(test_df.mt_mm, bins=15, color='c')
plt.title('Muscle thickness histogram')
plt.xlabel('mt_mm')
plt.ylabel('count')
plt.show()


In [ ]:
plt.hist(test_df.pa_deg, bins=15, color='y')
plt.title('Pennation angle histogram')
plt.xlabel('pa_deg')
plt.ylabel('count')
plt.show()


In [ ]:
plt.hist(test_df.fl_mm, bins=15, color='lightgreen')
plt.title('Fascicle length histogram')
plt.xlabel('fl_mm')
plt.ylabel('count')
plt.show()
